#Chapter 5 - Learning Embeddings and Representations
Before a neural network can recognize a digit, detect an object, interpret a sentence, translate text, or autocomplete an email, it has to do something more fundamental. It must decide which parts of the data matter for the task. That decision is not programmed into the model by hand. It is learned. The internal representations a network builds determine which structure it preserves, which variation it discounts, and ultimately what it can predict.


#Listing 5-1: Measuring concept closeness with cosine similarity
This listing uses a pretrained embedding model to convert sentences into dense vector representations. Cosine similarity is then used to measure how semantically related the resulting vectors are.

In [ ]:
# ------------------------------------------------------
# Step 1: Imports
# ------------------------------------------------------
from sentence_transformers import SentenceTransformer, util

# ------------------------------------------------------
# Step 2: Load sentence embedding model
# ------------------------------------------------------
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

# ------------------------------------------------------
# Step 3: Define example phrases
# ------------------------------------------------------
phrases = [
    "IBM Bob is an AI-powered IDE.",
    "An AI coding assistant helps programmers create applications.",
    "A chef prepared a gourmet five-course dinner in the kitchen."
]

# ------------------------------------------------------
# Step 4: Generate embeddings
# ------------------------------------------------------
embeddings = model.encode(
    phrases,
    convert_to_tensor=True,
    normalize_embeddings=True
)

# ------------------------------------------------------
# Step 5: Compute cosine similarities
# ------------------------------------------------------
sim_0_1 = util.cos_sim(embeddings[0], embeddings[1]).item()
sim_0_2 = util.cos_sim(embeddings[0], embeddings[2]).item()

# ------------------------------------------------------
# Step 6: Display results
# ------------------------------------------------------
print(f"Cosine Similarity (Phrase 1 vs Phrase 2): {sim_0_1:.4f}")
print(f"Cosine Similarity (Phrase 1 vs Phrase 3): {sim_0_2:.4f}")

# Listing 5-2: Reconstructing MNIST images with an autoencoder
This listing shows the core structure of a simple autoencoder trained on handwritten MNIST digits. The encoder compresses each 28 × 28 image into a low-dimensional bottleneck representation, and the decoder reconstructs the image from that compressed code.

In [ ]:
# ------------------------------------------------------
# Step 1: Imports and Setup
# ------------------------------------------------------
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------------------------------------------
# Step 2: Load MNIST
# ------------------------------------------------------
transform = transforms.ToTensor()
train_data = datasets.MNIST(
    root="data", train=True, transform=transform, download=True
)
train_loader = DataLoader(train_data, batch_size=256, shuffle=True)

# ------------------------------------------------------
# Step 3: Define a simple fully connected autoencoder
# ------------------------------------------------------
class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder: 784 -> 64 -> 16 (bottleneck)
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 64),
            nn.ReLU(),
            nn.Linear(64, 16),
            nn.ReLU(),
        )
        # Decoder: 16 -> 64 -> 784
        self.decoder = nn.Sequential(
            nn.Linear(16, 64),
            nn.ReLU(),
            nn.Linear(64, 28 * 28),
            nn.Sigmoid(),  # output pixels in [0, 1]
        )

        # ---------------------------------------------------
        # Improving the Output - To get sharper images,
        # comment out the original encoder and decoder above,
        # and uncomment this wider encoder and decoder:
        # ---------------------------------------------------
        # self.encoder = nn.Sequential(
        #     nn.Flatten(),
        #     nn.Linear(28 * 28, 128),
        #     nn.ReLU(),
        #     nn.Linear(128, 64), # 64-dimensional bottleneck
        #     nn.ReLU()
        # )
        # self.decoder = nn.Sequential(
        #     nn.Linear(64, 128),
        #     nn.ReLU(),
        #     nn.Linear(128, 28 * 28),
        #     nn.Sigmoid()
        # )

    def forward(self, x):
        z = self.encoder(x)
        out = self.decoder(z)
        return out.view(-1, 1, 28, 28)

model = Autoencoder().to(device)
criterion = nn.MSELoss() # MSE works well for grayscale
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# ------------------------------------------------------
# Step 4: Train the autoencoder to reconstruct its input
# ------------------------------------------------------
num_epochs = 3

for epoch in range(num_epochs):
    running_loss = 0.0
    for imgs, _ in train_loader:
        imgs = imgs.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, imgs)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch {epoch + 1}: loss {epoch_loss:.4f}")

# ------------------------------------------------------
# Step 5: Visualize original and reconstructed digits
# ------------------------------------------------------
model.eval()
imgs, _ = next(iter(train_loader))
imgs = imgs[:8].to(device)

with torch.no_grad():
    decoded = model(imgs).cpu()

imgs = imgs.cpu()

plt.figure(figsize=(12, 3))
for i in range(8):
    # Original
    ax = plt.subplot(2, 8, i + 1)
    plt.imshow(imgs[i].squeeze(), cmap="gray")
    ax.axis("off")

    # Reconstructed
    ax = plt.subplot(2, 8, i + 9)
    plt.imshow(decoded[i].squeeze(), cmap="gray")
    ax.axis("off")

plt.suptitle("Original digits (top) vs reconstructed digits (bottom)")
plt.tight_layout()
plt.show()


# Listing 5-3: Detecting anomalies with autoencoder reconstruction error
This program demonstrates anomaly detection with reconstruction error. The autoencoder trains only on digits 0 through 7, which define the normal baseline. During evaluation, digits 8 and 9 are reintroduced as structurally unfamiliar examples.

In [ ]:
# ------------------------------------------------------
# Step 1: Imports and Setup
# ------------------------------------------------------
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ------------------------------------------------------------------
# Step 2: Define a simple fully connected autoencoder
# ------------------------------------------------------------------
class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder: 784 -> 64 -> 16 (bottleneck)
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 64),
            nn.ReLU(),
            nn.Linear(64, 16),
            nn.ReLU(),
        )
        # Decoder: 16 -> 64 -> 784
        self.decoder = nn.Sequential(
            nn.Linear(16, 64),
            nn.ReLU(),
            nn.Linear(64, 28 * 28),
            nn.Sigmoid(),  # output pixels in [0, 1]
        )

    def forward(self, x):
        z = self.encoder(x)
        out = self.decoder(z)
        return out.view(-1, 1, 28, 28)

# ------------------------------------------------------------------
# Step 3: Load MNIST train and test sets
# ------------------------------------------------------------------
transform = transforms.ToTensor()

train_data = datasets.MNIST(
    root="data",
    train=True,
    transform=transform,
    download=True,
)

test_data = datasets.MNIST(
    root="data",
    train=False,
    transform=transform,
    download=True,
)

# ------------------------------------------------------------------
# Step 4: Keep only digits 0–7 for training (normal data)
# ------------------------------------------------------------------
train_mask = train_data.targets < 8
x_train_normal = train_data.data[train_mask].float() / 255.0   # [N, 28, 28]
x_train_normal = x_train_normal.unsqueeze(1)                   # [N, 1, 28, 28]

normal_loader = DataLoader(x_train_normal, batch_size=256, shuffle=True)

# ------------------------------------------------------------------
# Step 5: Create and train a new autoencoder on normal digits only
# ------------------------------------------------------------------
model = Autoencoder().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for imgs in normal_loader:
        imgs = imgs.to(device)

        optimizer.zero_grad()
        recon = model(imgs)
        loss = criterion(recon, imgs)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)

    epoch_loss = running_loss / len(normal_loader.dataset)
    print(f"Epoch {epoch + 1}: loss {epoch_loss:.4f}")

# ------------------------------------------------------------------
# Step 6: Score the entire test set by reconstruction error
# ------------------------------------------------------------------
x_test = test_data.data.float() / 255.0   # [N, 28, 28]
x_test = x_test.unsqueeze(1)              # [N, 1, 28, 28]
y_test = test_data.targets                # digit labels

test_ds = TensorDataset(x_test, y_test)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

model.eval()
all_errors = []
all_labels = []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        recon = model(imgs)

        # Mean squared error per image, averaged over all pixels
        err = ((imgs - recon) ** 2).mean(dim=[1, 2, 3]).cpu()

        all_errors.append(err)
        all_labels.append(labels)

errors = torch.cat(all_errors).numpy()
labels = torch.cat(all_labels).numpy()

# ------------------------------------------------------------------
# Step 7: Compare reconstruction error for normal digits (0-7)
#         and unusual digits (8-9)
# ------------------------------------------------------------------
normal_mask = labels < 8
anom_mask = labels >= 8

# Use a threshold based on NORMAL data only (typical anomaly-detection practice)
threshold = np.percentile(errors[normal_mask], 95)

# Use shared bin edges so the histograms are directly comparable
bins = np.linspace(errors.min(), errors.max(), 41)

plt.figure(figsize=(8, 4))

# Option A: counts with a clearer y-label
plt.hist(
    errors[normal_mask],
    bins=bins,
    histtype="step",
    label="Digits 0–7 (test set)",
)
plt.hist(
    errors[anom_mask],
    bins=bins,
    histtype="step",
    linestyle="--",
    label="Digits 8–9 (test set)",
)

# If you prefer Option B, uncomment density=True in both hist calls
# and change ylabel.
# plt.hist(..., density=True, ...)
# plt.ylabel("Probability density")

plt.axvline(threshold, color="black", linestyle=":", label="Threshold (95th pct of digits 0–7)")

plt.xlabel("Reconstruction error (MSE per image)")
plt.ylabel("Number of test images (per bin)")
plt.title("Autoencoder anomaly detection on MNIST")
plt.legend()
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------
# Step 8: Confirm what contributes to the right tail
# ------------------------------------------------------------------
tail_mask = errors >= threshold

# (A) Tail composition by digit label
unique, counts = np.unique(labels[tail_mask], return_counts=True)
tail_counts = dict(zip(unique.tolist(), counts.tolist()))
total_tail = int(tail_mask.sum())

print(f"Images in the tail (error ≥ threshold): {total_tail}")
print("Tail breakdown by digit label:")
for d in range(10):
    c = tail_counts.get(d, 0)
    if c > 0:
        print(f"  Digit {d}: {c} ({c / total_tail:.1%})")

# (B) Show a few highest-error examples and their labels
k = 12
idx_sorted = np.argsort(errors)[::-1]
top_idx = idx_sorted[:k]

print("\nTop error examples (label, error):")
for i in top_idx:
    print(f"  {int(labels[i])}, {errors[i]:.4f}")

# Visualize those top-k images
fig, axs = plt.subplots(1, k, figsize=(1.2 * k, 2))
for ax, i in zip(axs, top_idx):
    ax.imshow(x_test[i].squeeze().numpy(), cmap="gray")
    ax.set_title(str(int(labels[i])))
    ax.axis("off")
plt.suptitle("Highest reconstruction errors (titles are true labels)")
plt.tight_layout()
plt.show()